# ACE A11 PoC (Layer3, Auto-K)

Runs the fixed ACE PoC pipeline end-to-end from `src/ace_pipeline.py`.

Outputs:
- `run_meta.json`
- `cluster_scores.csv`
- `concepts_auto/cluster_XXX/*`

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

# Works both from server path and local clones
CANDIDATES = [
    Path('/home/SpeakerRec/BioVoice'),
    Path.cwd().resolve(),
    Path.cwd().resolve().parents[1] if len(Path.cwd().resolve().parents) > 1 else Path.cwd().resolve(),
]
ROOT = None
for c in CANDIDATES:
    if (c / 'resnet_293' / 'ace' / 'ace_a11_poc' / 'src' / 'ace_pipeline.py').exists():
        ROOT = c
        break
if ROOT is None:
    raise RuntimeError('Could not locate BioVoice root. Set ROOT manually.')

ACE_DIR = ROOT / 'resnet_293' / 'ace' / 'ace_a11_poc'
SRC_DIR = ACE_DIR / 'src'
CFG_PATH = ACE_DIR / 'configs' / 'a11_ace_config.yaml'

sys.path.insert(0, str(SRC_DIR))
from ace_pipeline import run_ace_poc  # noqa: E402

print('ROOT =', ROOT)
print('ACE_DIR =', ACE_DIR)
print('CFG_PATH =', CFG_PATH)
print('Config exists =', CFG_PATH.exists())

In [ ]:
# Run pipeline
meta = run_ace_poc(CFG_PATH)
print(json.dumps(meta, indent=2))

In [ ]:
# Inspect output artifacts
out_dir = Path(meta['output_dir'])
print('Output dir:', out_dir)

score_csv = out_dir / 'cluster_scores.csv'
meta_json = out_dir / 'run_meta.json'
concepts_dir = out_dir / 'concepts_auto'

print('cluster_scores.csv exists:', score_csv.exists())
print('run_meta.json exists:', meta_json.exists())
print('concepts_auto exists:', concepts_dir.exists())

if score_csv.exists():
    import pandas as pd
    df = pd.read_csv(score_csv)
    display(df.head(10))

if concepts_dir.exists():
    concept_folders = sorted([p for p in concepts_dir.iterdir() if p.is_dir()])
    print('clusters exported:', len(concept_folders))
    for p in concept_folders[:10]:
        npy_count = len(list(p.glob('*.npy')))
        print(' ', p.name, 'npy=', npy_count)